# Haik.io — Entraînement du détecteur de formes (brique A)

Ce notebook entraîne un **YOLO11** qui détecte 8 formes (cercle, carré, rectangle, triangle, étoile, losange, pentagone, hexagone) sur le dataset synthétique du projet.

**Avant de commencer**
1. `Exécution` → `Modifier le type d'exécution` → **GPU T4**.
2. Colab → icône 🔑 *Secrets* : ajoute `GITHUB_TOKEN` (seulement si ton dépôt est privé) et active l'accès au notebook.
3. Ta branche doit être poussée sur GitHub (`git push -u origin feature/entrainement_model --tags`).

**Ordre :** installation → dataset → contrôle visuel → test rapide (2 époques) → entraînement complet → évaluation → test sur tes vrais dessins → export.

## 1. Environnement

In [ ]:
import torch
print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("⚠️ Pas de GPU : Exécution > Modifier le type d'exécution > GPU T4, puis relance.")

### Google Drive (sauvegarde des résultats)
Colab peut se déconnecter. Les résultats d'entraînement sont donc écrits sur ton Drive, ce qui permet de reprendre.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DOSSIER_DRIVE = "/content/drive/MyDrive/haik"
RUNS = f"{DOSSIER_DRIVE}/runs"
os.makedirs(RUNS, exist_ok=True)
print("Résultats sauvegardés dans :", RUNS)

### Récupération du code
⚠️ Modifie `DEPOT` avec ton compte GitHub et le nom de ton dépôt.

In [ ]:
import os, subprocess
from google.colab import userdata

DEPOT = "TON-COMPTE/Haik.io"            # <-- À MODIFIER
BRANCHE = "feature/entrainement_model"

try:
    url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/{DEPOT}.git"
except Exception:
    url = f"https://github.com/{DEPOT}.git"      # dépôt public

if not os.path.exists("/content/Haik.io"):
    r = subprocess.run(["git", "clone", "-b", BRANCHE, url, "/content/Haik.io"], capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit("Échec du clonage : vérifie DEPOT, BRANCHE (poussée ?) et le secret GITHUB_TOKEN.")
os.chdir("/content/Haik.io")
print("Dossier de travail :", os.getcwd())

In [ ]:
!pip install -q ultralytics dvc pyyaml

## 2. Dataset
Le dataset de formes est **reproductible** : `dvc repro` le régénère à partir de `params.yaml` (≈ 8 min), sans rien télécharger.
Les pixels JPEG peuvent différer légèrement de ceux de ton PC (autre système), mais les formes, couleurs et positions sont les mêmes.

In [ ]:
!dvc repro
!git checkout -- dvc.lock reports 2>/dev/null; true

In [ ]:
import json
print(json.dumps(json.load(open("reports/dataset_stats.json")), indent=2, ensure_ascii=False)[:1500])

### Contrôle visuel
Chaque forme doit avoir une boîte et la bonne étiquette. Si ce n'est pas le cas, **ne lance pas l'entraînement**.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
from src.data.generer_formes import apercu

apercu(Path("data/formes"), nb=12)
display(Image("data/formes/apercu.jpg"))
os.remove("data/formes/apercu.jpg")

## 3. Paramètres d'entraînement (lus dans `params.yaml`)

In [ ]:
import yaml
P = yaml.safe_load(open("params.yaml"))["entrainement_formes"]
DATA = os.path.abspath("data/formes/data.yaml")
print(P)

## 4. Test rapide (2 minutes)
On entraîne 2 époques sur 5 % des données. Ça vérifie que tout fonctionne avant de lancer plusieurs heures d'entraînement.

In [ ]:
from ultralytics import YOLO

test = YOLO(P["modele"])
test.train(data=DATA, epochs=2, fraction=0.05, imgsz=P["imgsz"], batch=P["batch"],
           project=RUNS, name="smoke", exist_ok=True, workers=2, plots=False)

## 5. Entraînement complet
- Durée indicative : **1 à 3 h** sur T4 selon le modèle (`yolo11n` plus rapide, `yolo11m` plus lent). L'arrêt anticipé (`patience`) coupe l'entraînement quand le modèle ne progresse plus.
- Les poids sont sauvegardés à chaque époque dans ton Drive. Si Colab se déconnecte, passe à la cellule « Reprise ».

In [ ]:
model = YOLO(P["modele"])
model.train(
    data=DATA,
    epochs=P["epochs"],
    patience=P.get("patience", 15),
    imgsz=P["imgsz"],
    batch=P["batch"],
    project=RUNS, name="formes_v1", exist_ok=True,
    workers=2,
    cache="disk",       # accélère la lecture des 18 000 images
    cos_lr=True,
    flipud=0.5,         # une forme à l'envers reste la même forme
    seed=0,
)

### Reprise après une déconnexion (à lancer seulement si besoin)

In [ ]:
# from ultralytics import YOLO
# YOLO(f"{RUNS}/formes_v1/weights/last.pt").train(resume=True)

## 6. Évaluation sur le jeu de **test** (images jamais vues)

À regarder :
- **mAP50** : détecte-t-il les bonnes formes au bon endroit ? Vise **> 0,95** sur ces données synthétiques.
- **mAP50-95** : précision des boîtes (plus exigeant). Vise **> 0,80**.
- **Par classe** : une forme nettement plus faible que les autres (souvent pentagone / hexagone / étoile) indique qu'il faut plus de variété dans le générateur.
- **Matrice de confusion** : quelles formes sont confondues entre elles ?

⚠️ Un bon score ici ne prouve rien sur le monde réel. Le vrai test est la section 7.

In [ ]:
best = YOLO(f"{RUNS}/formes_v1/weights/best.pt")
m = best.val(data=DATA, split="test", imgsz=P["imgsz"], plots=True,
             project=RUNS, name="formes_v1_test", exist_ok=True)

print(f"mAP50     : {m.box.map50:.3f}")
print(f"mAP50-95  : {m.box.map:.3f}")
print("\nmAP50-95 par forme :")
for i, nom in m.names.items():
    print(f"  {nom:<10} {m.box.maps[i]:.3f}")

In [ ]:
from IPython.display import Image, display
for f in ["results.png"]:
    display(Image(f"{RUNS}/formes_v1/{f}"))
display(Image(f"{RUNS}/formes_v1_test/confusion_matrix_normalized.png"))

In [ ]:
import json
os.makedirs("reports", exist_ok=True)
metriques = {
    "modele": P["modele"], "epochs_max": P["epochs"], "imgsz": P["imgsz"],
    "test_map50": round(float(m.box.map50), 4),
    "test_map50_95": round(float(m.box.map), 4),
    "test_map50_95_par_forme": {nom: round(float(m.box.maps[i]), 4) for i, nom in m.names.items()},
}
json.dump(metriques, open("reports/entrainement_formes.json", "w"), indent=2, ensure_ascii=False)
import shutil; shutil.copy("reports/entrainement_formes.json", f"{DOSSIER_DRIVE}/entrainement_formes.json")
print(json.dumps(metriques, indent=2, ensure_ascii=False))

## 7. Test sur tes vrais dessins (le test qui compte)
Envoie 10 à 30 captures d'écran de **ton interface** (dessins faits à la main, avec ta palette) et quelques photos. Les erreurs ici montrent ce qui manque au dataset.

In [ ]:
from google.colab import files
import shutil
shutil.rmtree("reel", ignore_errors=True); os.makedirs("reel")
for nom, contenu in files.upload().items():
    open(f"reel/{nom}", "wb").write(contenu)

In [ ]:
from IPython.display import Image, display
import glob
res = best.predict(source="reel", conf=0.25, imgsz=P["imgsz"], save=True,
                   project="/content/pred", name="reel", exist_ok=True)
for f in sorted(glob.glob("/content/pred/reel/*"))[:12]:
    display(Image(f, width=480))

## 8. Export
Le modèle `best.pt` est copié sur ton Drive. Télécharge-le ensuite dans le dossier `models/` de ton projet, puis versionne-le avec DVC :

```powershell
dvc add models/formes_v1.pt
git add models/formes_v1.pt.dvc models/.gitignore reports/entrainement_formes.json
git commit -m "Modèle formes v1"
dvc push
```

In [ ]:
import shutil
os.makedirs(f"{DOSSIER_DRIVE}/models", exist_ok=True)
shutil.copy(f"{RUNS}/formes_v1/weights/best.pt", f"{DOSSIER_DRIVE}/models/formes_v1.pt")
print("Modèle copié sur le Drive :", f"{DOSSIER_DRIVE}/models/formes_v1.pt")

from google.colab import files
files.download(f"{RUNS}/formes_v1/weights/best.pt")